# Case Study 2 — Cross-technique results summary (EXP-3 through EXP-8)

## 1. Clone or refresh repository, and locate local output root

In [ ]:
import urllib.request
import zipfile
import shutil
from pathlib import Path
import sys

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "simo"  # Case Study 2 (and its case_study_1.confidence_intervals dependency) lives here

WORKSPACE_ROOT = Path.cwd()
REPO_ROOT = WORKSPACE_ROOT / "DiverseVul--IS-Project"
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

if REPO_ROOT.exists():
    print(f"Removing existing repository at {REPO_ROOT}...")
    shutil.rmtree(REPO_ROOT)

print(f"Downloading repository (branch: {REPO_BRANCH}) without git...")
clean_url = REPO_URL.removesuffix(".git")
zip_url = f"{clean_url}/archive/refs/heads/{REPO_BRANCH}.zip"
zip_path = Path.cwd() / "repo_temp.zip"

urllib.request.urlretrieve(zip_url, zip_path)

print("Extracting files...")
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(Path.cwd())

repo_name = clean_url.split("/")[-1]
extracted_folder = Path.cwd() / f"{repo_name}-{REPO_BRANCH}"
if extracted_folder.exists():
    extracted_folder.rename(REPO_ROOT)
zip_path.unlink()

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Same local layout every EXP-3..EXP-8 notebook writes its outputs to --
# this notebook reads FROM the VM's local filesystem, not from Drive.
DATA_ROOT = WORKSPACE_ROOT / "IntelligentSystemProject" / "VulnerabilityDetectionData"
OUTPUT_ROOT = DATA_ROOT / "outputs" / "case_study_2"

print("Repository:", REPO_ROOT)
print("Project directory:", PROJECT_DIR)
print("OUTPUT_ROOT (local VM):", OUTPUT_ROOT)


## 2. Locate saved experiment output directories

Same six-experiment matrix discussed throughout the project: two backbones (CodeBERTa, NeoBERT) x three methods (frozen linear probe, LoRA, HEFT).

In [ ]:
EXP3_DIR = OUTPUT_ROOT / "exp3_codeberta_linear_probe_v1"
EXP4_DIR = OUTPUT_ROOT / "exp4_codeberta_lora_v1"
EXP5_DIR = OUTPUT_ROOT / "exp5_codeberta_heft_v1"
EXP6_DIR = OUTPUT_ROOT / "exp6_neobert_linear_probe_v1"
EXP7_DIR = OUTPUT_ROOT / "exp7_neobert_lora_v1"
EXP8_DIR = OUTPUT_ROOT / "exp8_neobert_heft_v1"

RESULTS_OUTPUT_DIR = OUTPUT_ROOT / "cs2_results_summary"

EXPERIMENT_DIRS = {
    "EXP-3 (Linear Probe / CodeBERTa)": EXP3_DIR,
    "EXP-4 (LoRA / CodeBERTa)": EXP4_DIR,
    "EXP-5 (HEFT / CodeBERTa)": EXP5_DIR,
    "EXP-6 (Linear Probe / NeoBERT)": EXP6_DIR,
    "EXP-7 (LoRA / NeoBERT)": EXP7_DIR,
    "EXP-8 (HEFT / NeoBERT)": EXP8_DIR,
}

missing = {name: d for name, d in EXPERIMENT_DIRS.items() if not d.exists()}
if missing:
    print("Missing experiment output directories (run that notebook first):")
    for name, d in missing.items():
        print(f"  - {name}: {d}")
    raise FileNotFoundError("One or more experiment output directories were not found; see above.")

RESULTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("All six experiment output directories found.")
print("Results will be saved to:", RESULTS_OUTPUT_DIR)


## 3. Define a robust predictions loader

In [ ]:
import pandas as pd

def load_predictions(directory, pattern):
    matches = sorted(directory.glob(pattern))
    if not matches:
        matches = sorted(directory.glob(pattern.replace(".parquet", ".csv")))
    if not matches:
        raise FileNotFoundError(f"No file matching \'{pattern}\' found in {directory}")
    if len(matches) > 1:
        raise RuntimeError(f"Multiple files matching \'{pattern}\' found in {directory}: {matches}")
    path = matches[0]
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)


## 4. Load development (pooled nested-CV) and holdout predictions for all six experiments

Same naming convention everywhere (`exp{N}_nested_oof_predictions.parquet`, `exp{N}_holdout_predictions.csv`), so the loader is identical across all six -- no per-experiment special-casing needed.

In [ ]:
exp3_dev_predictions = load_predictions(EXP3_DIR, "*oof_predictions*.parquet")
exp3_holdout_predictions = load_predictions(EXP3_DIR, "*holdout_predictions*.parquet")

exp4_dev_predictions = load_predictions(EXP4_DIR, "*oof_predictions*.parquet")
exp4_holdout_predictions = load_predictions(EXP4_DIR, "*holdout_predictions*.parquet")

exp5_dev_predictions = load_predictions(EXP5_DIR, "*oof_predictions*.parquet")
exp5_holdout_predictions = load_predictions(EXP5_DIR, "*holdout_predictions*.parquet")

exp6_dev_predictions = load_predictions(EXP6_DIR, "*oof_predictions*.parquet")
exp6_holdout_predictions = load_predictions(EXP6_DIR, "*holdout_predictions*.parquet")

exp7_dev_predictions = load_predictions(EXP7_DIR, "*oof_predictions*.parquet")
exp7_holdout_predictions = load_predictions(EXP7_DIR, "*holdout_predictions*.parquet")

exp8_dev_predictions = load_predictions(EXP8_DIR, "*oof_predictions*.parquet")
exp8_holdout_predictions = load_predictions(EXP8_DIR, "*holdout_predictions*.parquet")

for name, dev, holdout in [
    ("EXP-3", exp3_dev_predictions, exp3_holdout_predictions),
    ("EXP-4", exp4_dev_predictions, exp4_holdout_predictions),
    ("EXP-5", exp5_dev_predictions, exp5_holdout_predictions),
    ("EXP-6", exp6_dev_predictions, exp6_holdout_predictions),
    ("EXP-7", exp7_dev_predictions, exp7_holdout_predictions),
    ("EXP-8", exp8_dev_predictions, exp8_holdout_predictions),
]:
    print(f"{name} dev rows: {len(dev)} | holdout rows: {len(holdout)}")


## 5. Validate that all six experiments share the same frozen holdout partition

Critical for the paired bootstrap comparisons below to be valid: every experiment must have been scored against the exact same held-out projects.

In [ ]:
holdout_project_sets = {
    "EXP-3": set(exp3_holdout_predictions["project"].unique()),
    "EXP-4": set(exp4_holdout_predictions["project"].unique()),
    "EXP-5": set(exp5_holdout_predictions["project"].unique()),
    "EXP-6": set(exp6_holdout_predictions["project"].unique()),
    "EXP-7": set(exp7_holdout_predictions["project"].unique()),
    "EXP-8": set(exp8_holdout_predictions["project"].unique()),
}

reference = holdout_project_sets["EXP-3"]
for name, projects in holdout_project_sets.items():
    if projects != reference:
        raise ValueError(
            f"{name} holdout projects differ from EXP-3\'s holdout projects."
        )

print("All six experiments share the identical", len(reference), "-project outer holdout.")


## 6. Compute project-block bootstrap confidence intervals per experiment

In [ ]:
import case_study_1.confidence_intervals as confidence_intervals

experiment_predictions = {
    "EXP-3 (Linear Probe / CodeBERTa)": {"dev": exp3_dev_predictions, "holdout": exp3_holdout_predictions},
    "EXP-4 (LoRA / CodeBERTa)": {"dev": exp4_dev_predictions, "holdout": exp4_holdout_predictions},
    "EXP-5 (HEFT / CodeBERTa)": {"dev": exp5_dev_predictions, "holdout": exp5_holdout_predictions},
    "EXP-6 (Linear Probe / NeoBERT)": {"dev": exp6_dev_predictions, "holdout": exp6_holdout_predictions},
    "EXP-7 (LoRA / NeoBERT)": {"dev": exp7_dev_predictions, "holdout": exp7_holdout_predictions},
    "EXP-8 (HEFT / NeoBERT)": {"dev": exp8_dev_predictions, "holdout": exp8_holdout_predictions},
}

ci_results = {}
for name, partitions in experiment_predictions.items():
    ci_results[name] = {
        split: confidence_intervals.bootstrap_metric_ci(
            frame,
            metric="average_precision_pr_auc",
            n_bootstrap=1000,
            random_state=42,
        )
        for split, frame in partitions.items()
    }
    for split, result in ci_results[name].items():
        print(f"{name} [{split}]")
        print(confidence_intervals.format_ci_report(result))
        print()


## 7. Summary table of PR-AUC with confidence intervals

In [ ]:
summary_rows = []
for name, splits in ci_results.items():
    row = {"experiment": name}
    for split, result in splits.items():
        row[f"{split}_pr_auc"] = result.point_estimate
        row[f"{split}_ci_low"] = result.ci_low
        row[f"{split}_ci_high"] = result.ci_high
    summary_rows.append(row)

summary_table = pd.DataFrame(summary_rows)
summary_table


## 8. Bar chart: development vs. holdout PR-AUC across all six techniques

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

names = list(experiment_predictions.keys())
x = np.arange(len(names))
width = 0.35

dev_points = [ci_results[n]["dev"].point_estimate for n in names]
dev_low = [ci_results[n]["dev"].point_estimate - ci_results[n]["dev"].ci_low for n in names]
dev_high = [ci_results[n]["dev"].ci_high - ci_results[n]["dev"].point_estimate for n in names]

holdout_points = [ci_results[n]["holdout"].point_estimate for n in names]
holdout_low = [ci_results[n]["holdout"].point_estimate - ci_results[n]["holdout"].ci_low for n in names]
holdout_high = [ci_results[n]["holdout"].ci_high - ci_results[n]["holdout"].point_estimate for n in names]

fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(x - width / 2, dev_points, width, yerr=[dev_low, dev_high], capsize=4, label="Development (pooled nested CV)")
ax.bar(x + width / 2, holdout_points, width, yerr=[holdout_low, holdout_high], capsize=4, label="Outer holdout")

# Visually separate the two backbones (CodeBERTa: EXP-3/4/5, NeoBERT: EXP-6/7/8)
ax.axvline(2.5, color="gray", linestyle=":", linewidth=1)
ax.text(1, ax.get_ylim()[1], "CodeBERTa", ha="center", va="bottom", fontsize=9, color="gray")
ax.text(4, ax.get_ylim()[1], "NeoBERT", ha="center", va="bottom", fontsize=9, color="gray")

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha="right")
ax.set_ylabel("PR-AUC")
ax.set_title("Case Study 2: development vs. holdout PR-AUC across all six techniques")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_OUTPUT_DIR / "cs2_all_techniques_pr_auc_ci.png")
plt.show()


## 9. Precision-recall curves on the outer holdout

In [ ]:
from sklearn.metrics import precision_recall_curve

fig, ax = plt.subplots(figsize=(8, 7))
for name, frame in [
    ("EXP-3 (Linear Probe / CodeBERTa)", exp3_holdout_predictions),
    ("EXP-4 (LoRA / CodeBERTa)", exp4_holdout_predictions),
    ("EXP-5 (HEFT / CodeBERTa)", exp5_holdout_predictions),
    ("EXP-6 (Linear Probe / NeoBERT)", exp6_holdout_predictions),
    ("EXP-7 (LoRA / NeoBERT)", exp7_holdout_predictions),
    ("EXP-8 (HEFT / NeoBERT)", exp8_holdout_predictions),
]:
    precision, recall, _ = precision_recall_curve(frame["label"], frame["y_score"])
    pr_auc = ci_results[name]["holdout"].point_estimate
    ax.plot(recall, precision, label=f"{name} (PR-AUC={pr_auc:.4f})")

positive_rate = exp3_holdout_predictions["label"].mean()
ax.axhline(positive_rate, linestyle="--", color="gray", label=f"Random baseline ({positive_rate:.4f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Outer holdout precision-recall curves, all six techniques")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS_OUTPUT_DIR / "cs2_all_techniques_pr_curves_holdout.png")
plt.show()


## 10. Paired bootstrap comparisons between techniques on the outer holdout

Only the nine comparisons that answer a specific question, not all fifteen possible pairs (see project discussion): three "does method complexity help, holding the backbone fixed" comparisons per backbone, plus three "does the backbone matter, holding the method fixed" comparisons.

In [ ]:
pairs = [
    # Does method complexity help, within CodeBERTa?
    ("EXP-3 (Linear Probe / CodeBERTa)", "EXP-4 (LoRA / CodeBERTa)"),
    ("EXP-3 (Linear Probe / CodeBERTa)", "EXP-5 (HEFT / CodeBERTa)"),
    ("EXP-4 (LoRA / CodeBERTa)", "EXP-5 (HEFT / CodeBERTa)"),
    # Does method complexity help, within NeoBERT?
    ("EXP-6 (Linear Probe / NeoBERT)", "EXP-7 (LoRA / NeoBERT)"),
    ("EXP-6 (Linear Probe / NeoBERT)", "EXP-8 (HEFT / NeoBERT)"),
    ("EXP-7 (LoRA / NeoBERT)", "EXP-8 (HEFT / NeoBERT)"),
    # Does the backbone matter, holding the method fixed?
    ("EXP-3 (Linear Probe / CodeBERTa)", "EXP-6 (Linear Probe / NeoBERT)"),
    ("EXP-4 (LoRA / CodeBERTa)", "EXP-7 (LoRA / NeoBERT)"),
    ("EXP-5 (HEFT / CodeBERTa)", "EXP-8 (HEFT / NeoBERT)"),
]

paired_results = {}
for name_a, name_b in pairs:
    result = confidence_intervals.paired_bootstrap_metric_ci(
        experiment_predictions[name_a]["holdout"],
        experiment_predictions[name_b]["holdout"],
        name_a,
        name_b,
        n_bootstrap=1000,
        random_state=42,
    )
    paired_results[(name_a, name_b)] = result
    print(confidence_intervals.format_paired_ci_report(result))
    print()


## 11. Save summary and comparison tables

In [ ]:
summary_table.to_csv(RESULTS_OUTPUT_DIR / "cs2_all_techniques_summary.csv", index=False)

paired_rows = [
    {
        "experiment_a": result.experiment_a,
        "experiment_b": result.experiment_b,
        "pr_auc_a": result.point_estimate_a,
        "pr_auc_b": result.point_estimate_b,
        "difference": result.point_estimate_diff,
        "ci_low": result.ci_low_diff,
        "ci_high": result.ci_high_diff,
        "significant": (result.ci_low_diff > 0) or (result.ci_high_diff < 0),
    }
    for result in paired_results.values()
]
paired_table = pd.DataFrame(paired_rows)
paired_table.to_csv(RESULTS_OUTPUT_DIR / "cs2_all_techniques_paired_holdout_comparisons.csv", index=False)
paired_table
